# SQLite avec Python : Analyse de bases de données

Notebook de l'article [SQLite avec Python : Analyse de bases de données](https://moncoachdata.com/blog/sqlite-python/) (MonCoachData). Exécute les cellules dans l'ordre.

Le code est testé avec Python 3.12 et Pandas 3 ; il fonctionne aussi sur Google Colab. Le notebook construit lui-même la base `flights.db` à partir des fichiers OpenFlights : rien à télécharger à la main.

Dans cet article, nous verrons comment utiliser `sqlite3` pour créer, interroger et mettre à jour des bases de données SQLite avec Python. Nous verrons également comment simplifier le travail avec les bases de données SQLite en utilisant la librairie [Pandas](https://pandas.pydata.org/). Le code est écrit pour Python 3.

Tout comme le format CSV, SQLite stocke les données dans un seul fichier, facile à partager. La plupart des langages de programmation offrent un bon support pour les bases de données SQLite.

Python ne fait pas exception : la librairie `sqlite3`, qui permet d'accéder aux bases SQLite, est incluse avec Python depuis la version 2.5. Pas besoin de l'installer.

**En bref :** la bibliothèque `sqlite3`, incluse avec Python, permet de créer, interroger et modifier une base SQLite : on se connecte avec `connect`, on crée un curseur, on exécute une requête avec `execute` puis on récupère les résultats avec `fetchall`. Avec Pandas, `read_sql_query` lit directement le résultat dans un DataFrame et `to_sql` crée une table. Pour les modifications, valide la transaction avec `commit` et utilise des requêtes paramétrées (`?`) pour éviter l'injection SQL.

## Quelles données utilise-t-on dans cet article ?

Avant de commencer, jetons un coup d'œil aux données sur lesquelles nous allons travailler : des données de vol de compagnies aériennes.

Elles contiennent des informations sur les compagnies aériennes, les aéroports et les routes entre les aéroports. Chaque itinéraire représente un vol répété qu'une compagnie aérienne effectue entre un aéroport de départ et un aéroport d'arrivée.

Toutes les données sont dans une base de données SQLite appelée `flights.db`, qui contient trois tables : `airports`, `airlines` et `routes`. Dans ce notebook, la cellule suivante construit cette base à partir des fichiers publics d'[OpenFlights](https://openflights.org/data.html) (dépôt GitHub), avec les mêmes colonnes que dans l'article.

## Construire la base flights.db

(Cette étape n'est nécessaire que dans ce notebook : dans l'article, la base est déjà prête.)

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

BASE = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/"

# On repart d'une base vide à chaque exécution
Path("flights.db").unlink(missing_ok=True)

airlines = pd.read_csv(BASE + "airlines.dat", header=None, dtype=str)
airlines.columns = ["id", "name", "alias", "iata", "icao", "callsign", "country", "active"]
airlines = airlines[airlines["id"] != "-1"].reset_index(drop=True)
airlines["id"] = airlines["id"].astype(int)

airports = pd.read_csv(BASE + "airports.dat", header=None, dtype=str)
airports.columns = ["id", "name", "city", "country", "code", "icao", "latitude", "longitude",
                    "altitude", "offset", "dst", "timezone", "type", "source"]
airports = airports.drop(columns=["type", "source"])
airports["id"] = airports["id"].astype(int)

routes = pd.read_csv(BASE + "routes.dat", header=None, dtype=str)
routes.columns = ["airline", "airline_id", "source", "source_id", "dest", "dest_id",
                  "codeshare", "stops", "equipment"]

conn = sqlite3.connect("flights.db")
airlines.to_sql("airlines", conn)
airports.to_sql("airports", conn)
routes.to_sql("routes", conn)
conn.close()

print(len(airlines), "compagnies,", len(airports), "aéroports,", len(routes), "routes")

### airlines

Voici 2 lignes de la table `airlines` :

In [ ]:
conn = sqlite3.connect("flights.db")
pd.read_sql_query("select * from airlines limit 2;", conn)

Comme tu peux le voir ci-dessus, chaque ligne représente une compagnie aérienne différente. Et chaque colonne est une propriété de cette compagnie aérienne, comme par exemple son nom et son pays. Chaque compagnie a aussi un identifiant unique (colonne `id`), ce qui permet de la retrouver facilement.

### airports

Voici 2 lignes de la table `airports` :

In [ ]:
pd.read_sql_query("select * from airports limit 2;", conn)

Comme tu peux le voir, chaque ligne correspond à un aéroport et contient des informations sur l'emplacement de l'aéroport. Chaque aéroport possède également un identifiant unique (`id`), ce qui permet de le retrouver facilement.

### routes

Voici 2 lignes de la table `routes` :

In [ ]:
pd.read_sql_query("select * from routes limit 2;", conn)

Chaque route contient un `airline_id`, l'ID de la compagnie aérienne qui dessert l'itinéraire, ainsi qu'un `source_id`, l'ID de l'aéroport d'origine, et un `dest_id`, l'ID de l'aéroport de destination.

In [ ]:
conn.close()

Maintenant que nous connaissons les données, commençons par nous connecter à la base de données et exécuter une requête.

## Comment exécuter une requête sur une base SQLite avec Python ?

Pour travailler avec une base de données SQLite avec Python, nous devons d'abord nous y connecter. Nous pouvons le faire avec la fonction [connect](https://docs.python.org/3/library/sqlite3.html?highlight=connect#sqlite3.connect), qui renvoie un objet [Connection](https://docs.python.org/3/library/sqlite3.html?highlight=connect#sqlite3.Connection):

In [ ]:
import sqlite3

conn = sqlite3.connect("flights.db")

Une fois que nous avons un objet Connection, nous pouvons ensuite créer un objet [Cursor](https://docs.python.org/3/library/sqlite3.html#cursor-objects). Les curseurs nous permettent d'exécuter des requêtes SQL sur une base de données :

In [ ]:
cur = conn.cursor()

Une fois que nous avons un objet Cursor, nous pouvons l'utiliser pour exécuter une requête sur la base de données avec la méthode [execute](https://docs.python.org/3/library/sqlite3.html?highlight=connect#sqlite3.Cursor.execute). Le code ci-dessous va chercher les 5 premières lignes de la table `airlines` :

In [ ]:
cur.execute("select * from airlines limit 5;");

Tu as peut-être remarqué que nous n'avons pas affecté le résultat de la requête à une variable : il faut une autre commande pour aller chercher les résultats. Nous utilisons la méthode [fetchall](https://docs.python.org/3/library/sqlite3.html?highlight=connect#sqlite3.Cursor.fetchall) pour récupérer tous les résultats d'une requête :

In [ ]:
results = cur.fetchall()
print(results)

Les résultats sont formatés comme une liste de [tuples](https://docs.python.org/3/tutorial/datastructures.html#tuples-and-sequences). Chaque tuple correspond à une ligne de la table `airlines` de `flights.db`.

Traiter les données de cette façon est peu intuitif : il faudrait ajouter manuellement les en-têtes de chaque colonne et analyser les données à la main. Heureusement, Pandas offre un moyen plus simple, que nous verrons dans la section suivante.

Avant de poursuivre, il est recommandé de fermer les objets Connection et Cursor ouverts. Cela empêche la base de données SQLite d'être verrouillée. Une base verrouillée ne peut plus être mise à jour, ce qui provoque parfois des erreurs.

Voici comment fermer le curseur et la connexion :

In [ ]:
cur.close()
conn.close()

## Comment cartographier les aéroports ?

En utilisant nos nouvelles connaissances sur les requêtes, nous pouvons créer un graphique qui montrera où se trouvent tous les aéroports dans le monde. Tout d'abord, nous extrayons les latitudes et les longitudes :

In [ ]:
import sqlite3

conn = sqlite3.connect("flights.db")
cur = conn.cursor()
coords = cur.execute("""
  select cast(longitude as float),
  cast(latitude as float)
  from airports;"""
).fetchall()

coords[:3]

La requête ci-dessus récupère les colonnes `latitude` et `longitude` de la table `airports` et les convertit en nombres flottants. Nous appelons ensuite la méthode `fetchall` pour les récupérer.

Nous devons ensuite configurer notre traçage en important [matplotlib](https://matplotlib.org/) (la bibliothèque de référence en Python pour tracer des graphiques). Combinée à la bibliothèque [Cartopy](https://scitools.org.uk/cartopy/docs/latest/), elle permet de créer des cartes uniquement avec Python. (L'article d'origine utilisait Basemap, qui n'est plus maintenu activement.)

Commençons par installer (si besoin) et importer les bibliothèques :

In [ ]:
%pip install -q cartopy

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

Ensuite, nous configurons la carte et dessinons les continents et les côtes qui formeront l'arrière-plan :

In [ ]:
fig = plt.figure(figsize=(14, 7))
ax = plt.axes(projection=ccrs.Mercator(min_latitude=-80, max_latitude=80))
ax.set_extent([-180, 180, -58, 80], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#eeeeee")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5);

Enfin, nous traçons les coordonnées de chaque aéroport sur la carte.

Nous avons récupéré une liste de tuples depuis la base SQLite : le premier élément de chaque tuple est la longitude de l'aéroport, le second sa latitude. Nous les convertissons en deux listes, puis nous les traçons sur la carte :

In [ ]:
# Le premier élément de chaque tuple est la longitude, le second la latitude
x = [longitude for longitude, latitude in coords]
y = [latitude for longitude, latitude in coords]

ax.scatter(x, y, s=2, color="red", transform=ccrs.PlateCarree())
fig

Nous obtenons une carte qui montre chaque aéroport dans le monde :

Comme tu l'as peut-être remarqué, travailler directement avec les résultats de `sqlite3` est pénible. Il fallait se souvenir de la position de chaque colonne dans les tuples, puis analyser les listes à la main.

Heureusement, Pandas offre un moyen plus facile de travailler avec les résultats des requêtes SQL.

## Comment lire les résultats d'une requête dans un DataFrame Pandas ?

Nous pouvons utiliser la fonction [read_sql_query](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html) de Pandas pour lire les résultats d'une requête SQL directement dans un DataFrame.

Le code ci-dessous exécute la même requête que tout à l'heure, mais retourne un DataFrame. Cette approche présente plusieurs avantages :

In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("flights.db")
df = pd.read_sql_query("select * from airlines limit 5;", conn)
df

- pas besoin de créer un objet Cursor ni d'appeler `fetchall` ;
- les noms des colonnes sont lus automatiquement ;
- on obtient un DataFrame, pour explorer rapidement les données.

Nous obtenons un DataFrame bien formaté, dont nous pouvons facilement manipuler les colonnes :

In [ ]:
df["country"]

Je te recommande fortement d'utiliser `read_sql_query` dès que possible.

## Comment cartographier les itinéraires ?

Maintenant que nous savons lire des requêtes dans des DataFrames, nous pouvons créer une carte de chaque itinéraire aérien dans le monde. Nous commençons par interroger la base de données. La requête ci-dessous va :

- obtenir la latitude et la longitude de l'aéroport source de chaque itinéraire ;
- obtenir la latitude et la longitude de l'aéroport de destination de chaque itinéraire ;
- convertir toutes les coordonnées en nombres flottants ;
- lire les résultats dans un DataFrame et les stocker dans la variable `routes`.

In [ ]:
routes = pd.read_sql_query("""
  select cast(sa.longitude as float) as source_lon,
  cast(sa.latitude as float) as source_lat,
  cast(da.longitude as float) as dest_lon,
  cast(da.latitude as float) as dest_lat
  from routes
  inner join airports sa on
  sa.id = routes.source_id
  inner join airports da on
  da.id = routes.dest_id;
  """,
  conn)

routes.head()

Nous configurons ensuite la carte :

In [ ]:
fig = plt.figure(figsize=(14, 7))
ax = plt.axes(projection=ccrs.Mercator(min_latitude=-80, max_latitude=80))
ax.set_extent([-180, 180, -58, 80], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="#eeeeee")
ax.add_feature(cfeature.COASTLINE, linewidth=0.5);

Nous parcourons ensuite les 3000 premières lignes pour les dessiner. Le code ci-dessous va :

- parcourir les 3000 premières lignes de la variable `routes` ;
- déterminer si l'itinéraire est trop long ;
- si l'itinéraire n'est pas trop long :

dessiner un arc (grand cercle) entre l'origine et la destination.

In [ ]:
for name, row in routes[:3000].iterrows():
    if abs(row["source_lon"] - row["dest_lon"]) < 90:
        # Dessine un grand cercle entre l'aéroport d'origine et celui d'arrivée.
        ax.plot([row["source_lon"], row["dest_lon"]], [row["source_lat"], row["dest_lat"]],
                color="b", linewidth=0.3, transform=ccrs.Geodetic())
fig

Nous obtenons la carte suivante :

C'est beaucoup plus efficace avec Pandas, qui transforme les résultats de la requête SQL en DataFrame, qu'avec les résultats bruts de `sqlite3`.

Maintenant que nous savons interroger la base de données, passons à sa modification.

## Comment modifier les lignes d'une base SQLite ?

Nous pouvons utiliser `sqlite3` pour modifier une base SQLite : insérer, mettre à jour ou supprimer des lignes. La connexion se crée comme précédemment, nous ignorons donc cette étape.

In [ ]:
cur.close()
conn.close()

conn = sqlite3.connect("flights.db")
cur = conn.cursor()

### Comment insérer des lignes avec Python ?

Pour insérer une ligne, nous écrivons une requête `INSERT`. Le code ci-dessous ajoute une nouvelle ligne à la table `airlines` : nous spécifions 9 valeurs, une pour chaque colonne de la table.

In [ ]:
cur = conn.cursor()
cur.execute("insert into airlines values (6161, 19846, 'Vol essai', '', '', null, null, null, 'Y')");

Si tu interroges la table depuis la même connexion, tu vois la nouvelle ligne. Mais depuis une autre connexion, tu ne la verras pas encore. En revanche, un fichier a été créé : `flights.db-journal`. Il stocke la modification jusqu'à ce que tu la valides (`commit`) dans la base principale `flights.db`.

In [ ]:
import os

# Depuis la même connexion, la ligne est déjà visible...
print(cur.execute("select * from airlines where id = 19846;").fetchall())

# ... mais pas depuis une autre connexion, tant que la transaction n'est pas validée
autre = sqlite3.connect("flights.db")
print(autre.execute("select * from airlines where id = 19846;").fetchall())
autre.close()

print("Fichier journal présent :", os.path.exists("flights.db-journal"))

SQLite n'écrit pas dans la base de données tant que tu n'as pas validé une [transaction](https://www.sqlite.org/lang_transaction.html). Une transaction regroupe une ou plusieurs requêtes dont les modifications sont appliquées à la base en une seule fois.

Cela facilite la récupération après une erreur ou une modification accidentelle : si l'une des requêtes échoue, la base n'est pas partiellement mise à jour (tout ou rien).

#### Exemple

Imagine deux tables : l'une contient les opérations sur les comptes bancaires (*opérations*), l'autre le montant en euros de chaque compte (*soldes*). Thomas, client de la banque, veut envoyer 50 € à sa sœur Émilie. Pour cela, la banque doit :

- créer une ligne d'opération indiquant que 50 € sont prélevés sur le compte de Thomas et envoyés à Émilie ;
- mettre à jour la ligne de Thomas dans la table des soldes, en retirant 50 € ;
- mettre à jour la ligne d'Émilie dans la table des soldes, en ajoutant 50 €.

Cela nécessite trois requêtes SQL distinctes. Si l'une échoue, la base contient des données incohérentes : par exemple, si les deux premières fonctionnent et que la troisième échoue, Thomas perd son argent mais Émilie ne le reçoit pas.

Avec une transaction, la base principale n'est mise à jour que si toutes les requêtes aboutissent. Le système ne peut donc pas se retrouver dans un état où des clients perdent leur argent.

Par défaut, `sqlite3` ouvre une transaction lorsque tu exécutes une requête qui modifie la base de données ([en savoir plus](https://docs.python.org/3/library/sqlite3.html#sqlite3-controlling-transactions)). Nous pouvons valider la transaction et ajouter notre nouvelle ligne à la table `airlines` avec la méthode [commit](https://docs.python.org/3/library/sqlite3.html#sqlite3.Connection.commit) :

In [ ]:
conn.commit()

Si nous interrogeons maintenant `flights.db`, nous voyons la ligne supplémentaire, qui contient notre compagnie d'essai :

In [ ]:
pd.read_sql_query("select * from airlines where id=19846;", conn)

### Comment passer des paramètres dans une requête SQLite ?

Dans la dernière requête, nous avons codé en dur les valeurs à insérer. La plupart du temps, ce sont des valeurs dynamiques, qui proviennent de données téléchargées ou d'une saisie utilisateur (sur un site web par exemple).

Avec des données dynamiques, il est tentant d'insérer les valeurs par mise en forme de chaîne Python :

In [ ]:
# MAUVAISE méthode : ne fais jamais cela
cur = conn.cursor()
name = "Vol Essai"
cur.execute("insert into airlines values (6162, 19847, '{0}', '', '', null, null, null, 'Y')".format(name))
conn.rollback()  # on annule : cette ligne ne doit pas rester dans la base

Il faut absolument éviter de faire cela : insérer des valeurs par mise en forme de chaîne rend ton programme vulnérable aux attaques par [injection SQL](https://fr.wikipedia.org/wiki/Injection_SQL).

Heureusement, `sqlite3` permet d'injecter des valeurs dynamiques sans formater de chaîne. Voici la bonne méthode :

In [ ]:
# BONNE méthode : requête paramétrée
cur = conn.cursor()
values = ("Vol Essai", "Y")
cur.execute("insert into airlines values (6162, 19847, ?, '', '', null, null, null, ?)", values)
conn.commit()

Chaque `?` de la requête est remplacé par une valeur de `values` : le premier par le premier élément, le deuxième par le deuxième, et ainsi de suite. Cela fonctionne pour n'importe quel type de requête.

Nous avons ainsi créé une **requête SQLite paramétrée**, qui évite les problèmes d'injection SQL.

### Comment mettre à jour des lignes ?

Nous pouvons modifier les lignes dans une table SQLite avec la méthode `execute` et une requête `UPDATE` :

In [ ]:
cur = conn.cursor()
values = ("France", 19847)
cur.execute("update airlines set country=? where id=?", values)
conn.commit()

Nous pouvons ensuite vérifier que la mise à jour a bien eu lieu :

In [ ]:
pd.read_sql_query("select * from airlines where id=19847;", conn)

### Comment supprimer des lignes ?

Enfin, nous pouvons supprimer des lignes avec la méthode `execute` et une requête `DELETE` :

In [ ]:
cur = conn.cursor()
values = (19847, )
cur.execute("delete from airlines where id=?", values)
conn.commit()

Nous vérifions ensuite que la suppression a eu lieu : aucune ligne ne correspond plus à notre requête :

In [ ]:
pd.read_sql_query("select * from airlines where id=19847;", conn)

## Comment créer une table SQLite avec Python ?

Nous pouvons créer des tables en exécutant une requête SQL. Créons une table qui représente chaque vol quotidien sur un itinéraire, avec les colonnes suivantes :

In [ ]:
cur = conn.cursor()
cur.execute("create table daily_flights (id integer, departure date, arrival date, number text, route_id integer)")
conn.commit()

- `id` : entier ;
- `departure` : date de départ du vol ;
- `arrival` : date d'arrivée du vol ;
- `number` : texte, le numéro de vol ;
- `route_id` : entier, l'identifiant de l'itinéraire emprunté.

Une fois la table créée, nous pouvons y insérer des données de façon classique :

In [ ]:
cur.execute("insert into daily_flights values (1, '2018-09-28 0:00', '2018-09-28 12:00', 'T1', 1)")
conn.commit()

Si nous interrogeons la table, nous voyons maintenant la ligne :

In [ ]:
pd.read_sql_query("select * from daily_flights;", conn)

### Comment créer une table avec Pandas ?

Pandas permet de créer des tables beaucoup plus rapidement : il suffit de créer un DataFrame, puis de l'exporter vers une table SQL. Commençons par créer le DataFrame :

In [ ]:
from datetime import datetime

df = pd.DataFrame(
    [[1, datetime(2018, 9, 29, 0, 0), datetime(2018, 9, 29, 12, 0), "T2", 1]],
    columns=["id", "departure", "arrival", "number", "route_id"]
)
df

Nous appelons ensuite la méthode [to_sql](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html) pour convertir `df` en table dans la base de données. Nous fixons le paramètre `if_exists` à `"replace"` pour remplacer toute table existante nommée `daily_flights` :

In [ ]:
df.to_sql("daily_flights", conn, if_exists="replace", index=False)

Nous vérifions que tout a fonctionné en interrogeant la base de données :

In [ ]:
pd.read_sql_query("select * from daily_flights;", conn)

## Comment modifier une table SQLite ?

Dans les cas réels de data science, l'une des difficultés est que les données d'une base changent souvent. Dans notre exemple, nous pourrions ajouter à la table des compagnies aériennes un champ indiquant le nombre d'avions de chaque compagnie (un nombre qui évolue au fil des mois).

Heureusement, SQLite permet de modifier une table pour y ajouter des colonnes :

In [ ]:
cur.execute("alter table airlines add column airplanes integer;");

Note que nous n'avons pas besoin d'appeler `commit` : les requêtes `ALTER TABLE` sont exécutées immédiatement et ne sont pas placées dans une transaction. Nous pouvons maintenant interroger la table et voir la colonne supplémentaire :

In [ ]:
pd.read_sql_query("select * from airlines limit 1;", conn)

Toutes les valeurs de cette colonne sont à `NULL` dans SQLite (`None` en Python), car elle n'a pas encore de valeurs.

### Comment modifier une table avec Pandas ?

Il est aussi possible de modifier une table avec Pandas : on la lit dans un DataFrame, on modifie le DataFrame, puis on l'exporte à nouveau vers une table :

In [ ]:
df = pd.read_sql("select * from daily_flights", conn)
df["delay_minutes"] = None
df.to_sql("daily_flights", conn, if_exists="replace", index=False)

In [ ]:
pd.read_sql_query("select * from daily_flights;", conn)

Le code ci-dessus ajoute une colonne `delay_minutes` à la table `daily_flights`.

In [ ]:
cur.close()
conn.close()